# File-Backed Larger Frequency Ranges

This notebook shows the current large-band pattern. Creating a
completely synthetic frame from scratch is still an eager operation:
the in-memory `Frame` owns a NumPy array. Once a large observation
exists on disk, `Frame.open(...)` and `Frame.open_copy(...)` let us
read, plot, reduce, and inject bounded regions without loading the
full spectrogram.

The point is to keep narrowband science operations local even when
the underlying observation covers a broad frequency range.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
def estimate_bytes(tchans, fchans, dtype=np.float32):
    return tchans * fchans * np.dtype(dtype).itemsize

hypothetical_fchans = 2**22
demo_fchans = 2**15
tchans = 12

print("hypothetical float32 size, GiB:", estimate_bytes(tchans, hypothetical_fchans) / 1024**3)
print("demo float64 scratch size, MiB:", estimate_bytes(tchans, demo_fchans, np.float64) / 1024**2)

In [ ]:
scratch = stg.Frame(
    tchans=tchans,
    fchans=demo_fchans,
    df=1 * u.Hz,
    dt=1 * u.s,
    fch1=(6e9 + demo_fchans - 1) * u.Hz,
    ascending=False,
    seed=31,
    source_name="Large-band scratch demo",
)
scratch.data[:] = 100.0
large_source = OUT / "large_band_source.h5"
scratch.save_hdf5(large_source)
print("wrote", large_source)

In [ ]:
with stg.Frame.open(large_source, mode="r") as backed:
    print("shape:", backed.shape)
    print("is file-backed:", backed.is_file_backed)
    print("full materialization would be MiB:", backed.tchans * backed.fchans * 8 / 1024**2)

    small = backed.read_frame(
        f_index_range=(demo_fchans // 2 - 64, demo_fchans // 2 + 64),
        t_index_range=(0, backed.tchans),
    )
    print("bounded region shape:", small.shape)

In [ ]:
large_injected = OUT / "large_band_injected.h5"
with stg.Frame.open_copy(
    large_source,
    large_injected,
    overwrite=True,
    max_chunk_bytes=4096,
) as backed:
    stats = backed.estimate_noise_stats(
        bounding_f_range=(
            backed.get_frequency(backed.fchans // 2 - 2),
            backed.get_frequency(backed.fchans // 2 + 2),
        ),
        t_index_range=(0, backed.tchans),
        config=stg.NoiseEstimationConfig(context_width=128, guard_width=8),
    )
    result = backed.add_signal(
        path=stg.constant_path(
            backed.get_frequency(backed.fchans // 2),
            drift_rate=0.25 * backed.unit_drift_rate,
        ),
        t_profile=stg.constant_t_profile(level=25),
        f_profile=stg.gaussian_f_profile(width=8 * backed.df),
        auto_bounding=True,
        truncate_below=1e-3,
    )
    print("noise stats:", stats)
    print("injection result:", result)

    fig, ax = plt.subplots(figsize=(9, 3))
    backed.plot(
        f_index_range=(backed.fchans // 2 - 80, backed.fchans // 2 + 80),
        t_index_range=(0, backed.tchans),
        db=False,
        colorbar=True,
    )
    ax.set_title("Patched region read back from file-backed output")
    display(fig)
    plt.close(fig)

In [ ]:
with stg.Frame.open(large_injected, mode="r", max_chunk_bytes=4096) as backed:
    spectrum = backed.spectrum(
        mode="sum",
        f_index_range=(backed.fchans // 2 - 128, backed.fchans // 2 + 128),
        max_chunk_bytes=4096,
    )
    ts = backed.timeseries(
        mode="mean",
        f_index_range=(backed.fchans // 2 - 128, backed.fchans // 2 + 128),
        max_chunk_bytes=4096,
    )

print("spectrum shape:", spectrum.shape)
print("time series shape:", ts.shape)
print("spectrum derived metadata:", spectrum.metadata["derived"])